In [1]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [2]:
with open("processed.pkl", "rb") as f:
    data = pickle.load(f)

print(data.keys())

dict_keys(['train_data', 'dev_data', 'test_data', 'train_arrays', 'dev_arrays', 'test_arrays', 'word2id', 'tag2id', 'id2tag', 'intent2id', 'id2intent', 'embedding_matrix'])


In [3]:
print("Train:", len(data["train_data"]))
print("Dev:", len(data["dev_data"]))
print("Test:", len(data["test_data"]))

assert len(data["train_data"]) == 11514
assert len(data["dev_data"]) == 2033
assert len(data["test_data"]) == 2974

Train: 11514
Dev: 2033
Test: 2974


In [4]:
example = data["dev_data"][0]

print(example)

{'id': '11', 'utt': 'turn the lights off please', 'tokens': ['turn', 'the', 'lights', 'off', 'please'], 'tags': ['O', 'O', 'O', 'O', 'O'], 'intent': 'iot_hue_lightoff', 'slots': []}


In [5]:
print("Sentence:", example["utt"])
print("Intent:", example["intent"])
print("Tokens:", example["tokens"])
print("Tags:", example["tags"])
print("Slots:", example["slots"])

Sentence: turn the lights off please
Intent: iot_hue_lightoff
Tokens: ['turn', 'the', 'lights', 'off', 'please']
Tags: ['O', 'O', 'O', 'O', 'O']
Slots: []


In [7]:
def intent_accuracy(gold, predicted):
    if len(gold) != len(predicted) or len(gold) == 0:
        raise ValueError("Answer lists must have equal, nonzero lengths")

    correct = 0

    for gold_label, predicted_label in zip(gold, predicted):
        if gold_label == predicted_label:
            correct += 1

    return 100 * correct / len(gold)

In [8]:
gold = [item["intent"] for item in data["dev_data"]]

print(intent_accuracy(gold, gold))

100.0


In [9]:
wrong = ["not_an_intent"] * len(gold)

print(intent_accuracy(gold, wrong))

0.0


In [10]:
slot_example = next(
    item for item in data["dev_data"] if item["slots"]
)

print("Sentence:", slot_example["utt"])
print("Tokens:", slot_example["tokens"])
print("Tags:", slot_example["tags"])
print("Slots:", slot_example["slots"])

Sentence: dim the lights in the hall
Tokens: ['dim', 'the', 'lights', 'in', 'the', 'hall']
Tags: ['O', 'O', 'O', 'O', 'O', 'B-house_place']
Slots: [('house_place', 'hall')]


In [11]:
def decode_slots(tokens, tags):
    if len(tokens) != len(tags):
        raise ValueError("Token and tag lengths differ")

    slots = []
    current_type = None
    current_words = []

    for token, tag in list(zip(tokens, tags)) + [(None, "O")]:
        if tag.startswith("I-") and tag[2:] == current_type:
            current_words.append(token)
            continue

        if current_type is not None:
            slots.append((current_type, " ".join(current_words)))

        if tag == "O":
            current_type = None
            current_words = []
        elif tag.startswith(("B-", "I-")) and len(tag) > 2:
            current_type = tag[2:]
            current_words = [token]
        else:
            raise ValueError("Invalid tag: " + tag)

    return slots


decoded = decode_slots(
    slot_example["tokens"],
    slot_example["tags"]
)

print(decoded)

assert decoded == [tuple(slot) for slot in slot_example["slots"]]

[('house_place', 'hall')]


In [12]:
from collections import Counter

def slot_micro_f1(gold_slots, predicted_slots):
    if len(gold_slots) != len(predicted_slots) or len(gold_slots) == 0:
        raise ValueError("Answer lists must have equal, nonzero lengths")

    tp = 0
    fp = 0
    fn = 0

    for gold, predicted in zip(gold_slots, predicted_slots):
        gold_counts = Counter(tuple(slot) for slot in gold)
        predicted_counts = Counter(tuple(slot) for slot in predicted)

        tp += sum((gold_counts & predicted_counts).values())
        fp += sum((predicted_counts - gold_counts).values())
        fn += sum((gold_counts - predicted_counts).values())

    denominator = 2 * tp + fp + fn
    f1 = 100 * 2 * tp / denominator if denominator else 0.0

    return {"slot_f1": f1, "tp": tp, "fp": fp, "fn": fn}


gold_slots = [item["slots"] for item in data["dev_data"]]

perfect = slot_micro_f1(gold_slots, gold_slots)
empty = slot_micro_f1(gold_slots, [[] for _ in gold_slots])

print("Perfect answers:", perfect)
print("No predicted slots:", empty)

assert perfect["slot_f1"] == 100.0
assert empty["slot_f1"] == 0.0

Perfect answers: {'slot_f1': 100.0, 'tp': 2012, 'fp': 0, 'fn': 0}
No predicted slots: {'slot_f1': 0.0, 'tp': 0, 'fp': 0, 'fn': 2012}


In [13]:
gold_case = [[("house_place", "hall")]]

cases = {
    "Correct": [[("house_place", "hall")]],
    "Wrong type": [[("place_name", "hall")]],
    "Wrong text": [[("house_place", "the hall")]],
    "Duplicate": [[("house_place", "hall"), ("house_place", "hall")]],
    "Missing": [[]]
}

expected = {
    "Correct": (1, 0, 0),
    "Wrong type": (0, 1, 1),
    "Wrong text": (0, 1, 1),
    "Duplicate": (1, 1, 0),
    "Missing": (0, 0, 1)
}

for name, prediction in cases.items():
    result = slot_micro_f1(gold_case, prediction)

    assert (result["tp"], result["fp"], result["fn"]) == expected[name]

    print(name, f'{result["slot_f1"]:.2f}%')

Correct 100.00%
Wrong type 0.00%
Wrong text 0.00%
Duplicate 66.67%
Missing 0.00%


In [14]:
word2id = data["word2id"]
tag2id = data["tag2id"]
id2tag = data["id2tag"]
intent2id = data["intent2id"]
id2intent = data["id2intent"]
embedding_matrix = data["embedding_matrix"]

print("Vocabulary size:", len(word2id))
print("BIO tags:", len(tag2id))
print("Intent labels:", len(intent2id))
print("Embedding shape:", embedding_matrix.shape)

assert len(tag2id) == 111
assert len(intent2id) == 60

for label, number in tag2id.items():
    assert id2tag[number] == label

for label, number in intent2id.items():
    assert id2intent[number] == label

Vocabulary size: 5304
BIO tags: 111
Intent labels: 60
Embedding shape: (5304, 300)


In [15]:
dev_arrays = data["dev_arrays"]

dev_x = torch.from_numpy(dev_arrays[0])
dev_tags = torch.from_numpy(dev_arrays[1])
dev_intents = torch.from_numpy(dev_arrays[2])
dev_lengths = torch.from_numpy(dev_arrays[3])

print("Input shape:", dev_x.shape)
print("Slot label shape:", dev_tags.shape)
print("Intent label shape:", dev_intents.shape)
print("Length shape:", dev_lengths.shape)

assert dev_x.shape == dev_tags.shape
assert len(dev_x) == len(dev_tags) == len(dev_intents) == len(dev_lengths) == 2033

Input shape: torch.Size([2033, 27])
Slot label shape: torch.Size([2033, 27])
Intent label shape: torch.Size([2033])
Length shape: torch.Size([2033])


In [16]:
for i, item in enumerate(data["dev_data"]):
    length = int(dev_lengths[i])

    assert length == len(item["tokens"])
    assert item["tokens"] == item["utt"].split()

    expected_words = [
        word2id.get(word, 1) for word in item["tokens"]
    ]

    assert dev_x[i, :length].tolist() == expected_words

    restored_tags = [
        id2tag[int(number)] for number in dev_tags[i, :length]
    ]

    assert restored_tags == item["tags"]
    assert id2intent[int(dev_intents[i])] == item["intent"]

    restored_slots = decode_slots(item["tokens"], restored_tags)

    assert restored_slots == [
        tuple(slot) for slot in item["slots"]
    ]

print("All 2033 dev examples passed the alignment checks.")

All 2033 dev examples passed the alignment checks.


In [18]:
class JointModel(nn.Module):
    def __init__(
        self,
        emb_matrix,
        n_tags,
        n_intents,
        hidden=256,
        layers=2,
        dropout=0.4
    ):
        super().__init__()

        self.emb = nn.Embedding.from_pretrained(
            torch.tensor(emb_matrix),
            freeze=False,
            padding_idx=0
        )

        self.emb_drop = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            emb_matrix.shape[1],
            hidden,
            num_layers=layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if layers > 1 else 0
        )

        self.drop = nn.Dropout(dropout)
        self.slot_head = nn.Linear(2 * hidden, n_tags)
        self.intent_head = nn.Linear(4 * hidden, n_intents)

    def forward(self, x, lengths):
        embeddings = self.emb_drop(self.emb(x))

        packed = pack_padded_sequence(
            embeddings,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            output,
            batch_first=True,
            total_length=x.size(1)
        )

        output = self.drop(output)
        slot_logits = self.slot_head(output)

        mask = (x != 0).unsqueeze(-1)

        maximum = output.masked_fill(
            ~mask, -1e9
        ).max(dim=1).values

        mean = (
            (output * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        )

        intent_logits = self.intent_head(
            torch.cat([maximum, mean], dim=-1)
        )

        return intent_logits, slot_logits

In [19]:
checkpoint_path = "tune_dropout0.5.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

model = JointModel(
    embedding_matrix,
    len(tag2id),
    len(intent2id),
    hidden=config["hidden"],
    layers=config["layers"],
    dropout=config["dropout"]
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

total_parameters = sum(
    parameter.numel() for parameter in model.parameters()
)

assert total_parameters <= 20_000_000

print("Checkpoint:", checkpoint_path)
print("Best epoch:", checkpoint["epoch"])
print("Dropout:", config["dropout"])
print("Total parameters:", total_parameters)
print("Saved dev intent accuracy:", checkpoint["val_metrics"]["intent_acc"])
print("Saved dev slot F1:", checkpoint["val_metrics"]["slot_f1"])

Checkpoint: tune_dropout0.5.pt
Best epoch: 40
Dropout: 0.5
Total parameters: 4429387
Saved dev intent accuracy: 86.07968519429414
Saved dev slot F1: 79.2896568274538


In [20]:
predicted_intents = []
predicted_slots = []

batch_size = 256

model.eval()

with torch.inference_mode():
    for start in range(0, len(dev_x), batch_size):
        end = min(start + batch_size, len(dev_x))

        lengths = dev_lengths[start:end]
        max_length = int(lengths.max())

        inputs = dev_x[start:end, :max_length].to(device)

        intent_logits, slot_logits = model(inputs, lengths)

        intent_ids = intent_logits.argmax(dim=-1).cpu()
        tag_ids = slot_logits.argmax(dim=-1).cpu()

        for j in range(end - start):
            item = data["dev_data"][start + j]
            length = int(lengths[j])

            intent = id2intent[int(intent_ids[j])]

            tags = [
                id2tag[int(number)]
                for number in tag_ids[j, :length]
            ]

            slots = decode_slots(item["tokens"], tags)

            predicted_intents.append(intent)
            predicted_slots.append(slots)

assert len(predicted_intents) == 2033
assert len(predicted_slots) == 2033

print("Dev predictions collected:", len(predicted_intents))

Dev predictions collected: 2033


In [21]:
gold_intents = [
    item["intent"] for item in data["dev_data"]
]

gold_slots = [
    item["slots"] for item in data["dev_data"]
]

dev_accuracy = intent_accuracy(
    gold_intents, predicted_intents
)

dev_slot_result = slot_micro_f1(
    gold_slots, predicted_slots
)

saved_accuracy = checkpoint["val_metrics"]["intent_acc"]
saved_slot_f1 = checkpoint["val_metrics"]["slot_f1"]

print(f"Calculated dev intent accuracy: {dev_accuracy:.2f}%")
print(f"Saved dev intent accuracy: {saved_accuracy:.2f}%")

print(f"Calculated dev slot F1: {dev_slot_result['slot_f1']:.2f}%")
print(f"Saved dev slot F1: {saved_slot_f1:.2f}%")

assert abs(dev_accuracy - saved_accuracy) < 0.000001
assert abs(dev_slot_result["slot_f1"] - saved_slot_f1) < 0.000001

print("Dev scores match the checkpoint records.")

Calculated dev intent accuracy: 86.08%
Saved dev intent accuracy: 86.08%
Calculated dev slot F1: 79.29%
Saved dev slot F1: 79.29%
Dev scores match the checkpoint records.


In [22]:
test_arrays = data["test_arrays"]

test_x = torch.from_numpy(test_arrays[0])
test_lengths = torch.from_numpy(test_arrays[3])

assert len(test_x) == len(test_lengths) == len(data["test_data"]) == 2974

test_predicted_intents = []
test_predicted_slots = []

model.eval()

with torch.inference_mode():
    for start in range(0, len(test_x), batch_size):
        end = min(start + batch_size, len(test_x))

        lengths = test_lengths[start:end]
        max_length = int(lengths.max())

        inputs = test_x[start:end, :max_length].to(device)

        intent_logits, slot_logits = model(inputs, lengths)

        intent_ids = intent_logits.argmax(dim=-1).cpu()
        tag_ids = slot_logits.argmax(dim=-1).cpu()

        for j in range(end - start):
            item = data["test_data"][start + j]
            length = int(lengths[j])

            assert length == len(item["tokens"])

            intent = id2intent[int(intent_ids[j])]

            tags = [
                id2tag[int(number)]
                for number in tag_ids[j, :length]
            ]

            slots = decode_slots(item["tokens"], tags)

            test_predicted_intents.append(intent)
            test_predicted_slots.append(slots)

assert len(test_predicted_intents) == 2974
assert len(test_predicted_slots) == 2974

test_gold_intents = [
    item["intent"] for item in data["test_data"]
]

test_gold_slots = [
    item["slots"] for item in data["test_data"]
]

test_accuracy = intent_accuracy(
    test_gold_intents, test_predicted_intents
)

test_slot_result = slot_micro_f1(
    test_gold_slots, test_predicted_slots
)

print(f"Test intent accuracy: {test_accuracy:.2f}%")
print(f"Test slot micro-F1: {test_slot_result['slot_f1']:.2f}%")

Test intent accuracy: 85.27%
Test slot micro-F1: 75.61%


In [23]:
import pandas as pd

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

results = pd.DataFrame([
    {
        "Model": "Stage-1",
        "Split": "Validation",
        "Total parameters": total_parameters,
        "Trainable parameters": trainable_parameters,
        "Intent accuracy (%)": dev_accuracy,
        "Slot micro-F1 (%)": dev_slot_result["slot_f1"],
        "Best epoch": checkpoint["epoch"]
    },
    {
        "Model": "Stage-1",
        "Split": "Test",
        "Total parameters": total_parameters,
        "Trainable parameters": trainable_parameters,
        "Intent accuracy (%)": test_accuracy,
        "Slot micro-F1 (%)": test_slot_result["slot_f1"],
        "Best epoch": checkpoint["epoch"]
    }
])

print("Checkpoint:", checkpoint_path)

display(results.style.format({
    "Intent accuracy (%)": "{:.2f}",
    "Slot micro-F1 (%)": "{:.2f}"
}))

Checkpoint: tune_dropout0.5.pt


,Model,Split,Total parameters,Trainable parameters,Intent accuracy (%),Slot micro-F1 (%),Best epoch
0,Stage-1,Validation,4429387,4429387,86.08,79.29,40
1,Stage-1,Test,4429387,4429387,85.27,75.61,40


In [24]:
import json
import hashlib
from pathlib import Path

evaluation_record = {
    "checkpoint": checkpoint_path,
    "checkpoint_sha256": hashlib.sha256(
        Path(checkpoint_path).read_bytes()
    ).hexdigest(),
    "config": checkpoint["config"],
    "best_epoch": int(checkpoint["epoch"]),
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
    "validation": {
        "intent_accuracy": dev_accuracy,
        **dev_slot_result
    },
    "test": {
        "intent_accuracy": test_accuracy,
        **test_slot_result
    }
}

with open("my_stage1_evaluation.json", "w") as f:
    json.dump(evaluation_record, f, indent=2)

results.to_csv(
    "my_stage1_results.csv",
    index=False,
    float_format="%.2f"
)

print("Saved my_stage1_evaluation.json")
print("Saved my_stage1_results.csv")

Saved my_stage1_evaluation.json
Saved my_stage1_results.csv


In [25]:
import ast

function_names = {
    "intent_accuracy",
    "decode_slots",
    "slot_micro_f1"
}

function_sources = {}

for cell_source in get_ipython().history_manager.input_hist_raw:
    try:
        tree = ast.parse(cell_source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            if node.name in function_names:
                function_sources[node.name] = ast.get_source_segment(
                    cell_source, node
                )

assert function_names == set(function_sources)

shared_code = "from collections import Counter\n\n"

for name in ["intent_accuracy", "decode_slots", "slot_micro_f1"]:
    shared_code += function_sources[name] + "\n\n"

Path("shared_evaluation.py").write_text(
    shared_code,
    encoding="utf-8"
)

print("Saved shared_evaluation.py")

Saved shared_evaluation.py
